# McKenzie et al. (2025) on dandiset 000552: reactivation in ripples (Figure 8)

Paper: McKenzie et al. (2025), eLife 14:RP105183, [doi:10.7554/elife.105183](https://doi.org/10.7554/elife.105183);
claims file `claims/mckenzie-2025.yaml`, written from Reviewed Preprint version 1.

Dataset: [DANDI:000552](https://dandiarchive.org/dandiset/000552/0.230630.2304) version **0.230630.2304**, the
version the paper cites (Methods, Electrophysiology subjects). Pyramidal cells are taken from the
CellExplorer labels of the later version 0.261113.1035, after checking that its spike times are the
same; the pinned version has no cell types. The sessions and their checks are in `settling.ipynb`.

| Claim | Here |
|---|---|
| fig8b-late-moments-reactivated | tested |
| fig8d-familiar-no-difference | tested |
| fig8-no-pre-reactivation | tested |

Method (Methods, Reactivation analysis; claims file E10): the ripples are those deposited in the files
(Huszar et al.; the broadband LFP is only in the raw files, 120 to 180 GB each). Each ripple's vector is
each pyramidal unit's spike count over the ripple divided by its duration; it is correlated with the
100 ms vectors of the run; the null moves each unit's value on a ripple to a random other ripple
(an independent permutation of each unit across the pool), 1000 times; a ripple reactivates a moment
when its R is above all 1000 shuffles (above their 99.9th percentile). The moments are the bins of
minute 1 (0 to 60 s) and of minutes 10 to 11 (600 to 660 s, Figure 8B), 1200 per run.
Readings: pre ripples (a) the 30 min before the run, (b) the first 30 min of the recording; post ripples
the 30 min after the run; track vectors z-scored over the first 1000 s or raw counts; shuffle pool
(a) the set the ripple belongs to, (b) pre and post together. The main reading is pre (a), z-scored, pool (a);
each other reading is run with the rest held at the main one (pre (b) is also run with pool (b)), not
fully crossed, to keep the notebook under half an hour.

In [ ]:
import time, zlib, warnings, resource
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import settle as S

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)
T_START = time.time()

def seed(claim, k=0):
    return zlib.crc32(f"mckenzie-2025/{claim}".encode()) + k

SESS = {}
for sid in S.SESSIONS:
    s = S.load_session(sid)
    if len(s["runs"]) < 2 or s["runs"][1]["t1"] - s["runs"][1]["t0"] < 660:
        continue                      # the selection of settling.ipynb (claims file E1)
    s["cell_type"], _ = S.cell_types(sid, s["spike_index"])
    SESS[sid] = s
TEST = list(SESS)
print(f"{len(TEST)} sessions loaded in {time.time() - T_START:.0f} s:", TEST)

In [ ]:
WIN = {}
rows = []
for sid, s in SESS.items():
    rec0 = 0.0 if s["epochs"] is None else float(s["epochs"][0, 0])
    for env, r in (("fam", s["runs"][0]), ("nov", s["runs"][1])):
        w = dict(preA=(r["t0"] - 1800, r["t0"]), preB=(rec0, rec0 + 1800), post=(r["t1"], r["t1"] + 1800))
        WIN[(sid, env)] = w
        row = dict(session=sid, env=env, t0=round(r["t0"]), t1=round(r["t1"]))
        for k, (a, b) in w.items():
            m = (s["rip"][:, 0] >= a) & (s["rip"][:, 0] < b)
            row[f"ripples_{k}"] = int(m.sum())
            row[f"{k}_window"] = f"{a:.0f}-{b:.0f}"
        # the post window must not run into the next maze run
        nxt = [x["t0"] for x in s["runs"] if x["t0"] > r["t0"]]
        row["post_overlaps_next_run_s"] = round(max(0.0, r["t1"] + 1800 - nxt[0]), 1) if nxt else 0.0
        row["pyramidal"] = int((s["cell_type"] == "Pyramidal Cell").sum())
        rows.append(row)
print(pd.DataFrame(rows).to_string())
print("ripple durations (ms):", {sid: (round(s['rip_dur'].min() * 1e3), round(s['rip_dur'].max() * 1e3)) for sid, s in SESS.items()})

In [ ]:
MOM = {}
for sid, s in SESS.items():
    pyr = np.where(s["cell_type"] == "Pyramidal Cell")[0]
    for env, r in (("fam", s["runs"][0]), ("nov", s["runs"][1])):
        V = S.run_vectors(s, r, pyr)
        sel = np.r_[np.where(V["t"] < 60)[0], np.where((V["t"] >= 600) & (V["t"] < 660))[0]]
        MOM[(sid, env)] = dict(units=V["units"], z=V["z"][sel], raw=V["counts"][sel], early=V["t"][sel] < 60)

def run_reactivation(sid, env, rng, pre="preA", vec="z", pool="a", sets=("pre", "post")):
    s = SESS[sid]; w = WIN[(sid, env)]; mo = MOM[(sid, env)]
    rv = {k: S.ripple_vectors(s, mo["units"], [w[pre if k == "pre" else "post"]]) for k in sets}
    out = {}
    if pool == "a":
        for k in sets:
            res = S.reactivation(rv[k], {k: np.ones(len(rv[k]), bool)}, mo[vec], rng)
            out[k] = res[k]
    else:
        allv = np.concatenate([rv[k] for k in sets])
        lab = np.concatenate([[k] * len(rv[k]) for k in sets])
        res = S.reactivation(allv, {k: lab == k for k in sets}, mo[vec], rng)
        out.update(res)
    rows = []
    for k, (frac, n) in out.items():
        rows.append(dict(session=sid, env=env, pre=pre, vec=vec, pool=pool, set=k, n_ripples=n,
                         early_pct=100 * np.nanmean(frac[mo["early"]]), late_pct=100 * np.nanmean(frac[~mo["early"]]),
                         all_pct=100 * np.nanmean(frac)))
    return rows

def readings(env, claim):
    rng = np.random.default_rng(seed(claim))
    rows = []
    for sid in TEST:
        # the main reading, and each other reading varied one at a time (the full cross would double the time)
        rows += run_reactivation(sid, env, rng, "preA", "z", "a", ("pre", "post"))
        rows += run_reactivation(sid, env, rng, "preB", "z", "a", ("pre",))
        rows += run_reactivation(sid, env, rng, "preA", "raw", "a", ("pre", "post"))
        rows += run_reactivation(sid, env, rng, "preA", "z", "b")
        rows += run_reactivation(sid, env, rng, "preB", "z", "b")
    return pd.DataFrame(rows)

def seeds_main(env, claim, sets=("post",)):
    rows = []
    for k in range(1, 5):
        rng = np.random.default_rng(seed(claim) + 1000 * k)
        for sid in TEST:
            for r in run_reactivation(sid, env, rng, "preA", "z", "a", sets):
                rows.append(dict(r, seed_offset=1000 * k))
    return pd.DataFrame(rows)

def late_vs_early(df):
    out = []
    for key, d in df[df.set == "post"].groupby(["pre", "vec", "pool"]):
        t, p = stats.ttest_rel(d.late_pct, d.early_pct)
        out.append(dict(zip(["pre", "vec", "pool"], key), early_mean=d.early_pct.mean(), late_mean=d.late_pct.mean(),
                        diff=(d.late_pct - d.early_pct).mean(), t=t, df=len(d) - 1, p=p))
    return pd.DataFrame(out)

## fig8b-late-moments-reactivated

> Moments 10 to 11 minutes after a novel context transition are more likely to be reactivated in post-experience ripples than moments 0 to 1 minutes after it.

Criterion: per session the mean over moments of the percentage of post ripples that reactivate each
moment, in 600 to 660 s and in 0 to 60 s; reproduced when late is higher by the paired t-test over
sessions at p < 0.05, under one combination of the readings.

First the false positive rate of the procedure, which the paper gives as 0.001: the same test applied to
ripple vectors whose units were permuted across ripples once beforehand (no ensemble structure left).

In [ ]:
rng = np.random.default_rng(seed("fig8b-late-moments-reactivated") + 1)
fp = []
for sid in TEST:
    s = SESS[sid]; mo = MOM[(sid, "nov")]
    rv = S.ripple_vectors(s, mo["units"], [WIN[(sid, "nov")]["post"]])
    res = S.reactivation(rng.permuted(rv, axis=0), {"x": np.ones(len(rv), bool)}, mo["z"], rng)
    fp.append(res["x"][0].mean())
print("false positive rate by session:", np.round(fp, 5), "mean", round(float(np.mean(fp)), 5))

In [ ]:
t0 = time.time()
R_NOV = readings("nov", "fig8b-late-moments-reactivated")
print(f"novel readings in {time.time() - t0:.0f} s")
print(R_NOV[R_NOV.set == "post"].round(3).to_string())
LV_NOV = late_vs_early(R_NOV)
print(LV_NOV.round(4).to_string())

In [ ]:
t0 = time.time()
SD_NOV = seeds_main("nov", "fig8b-late-moments-reactivated")
main = R_NOV[(R_NOV.set == "post") & (R_NOV.pre == "preA") & (R_NOV.vec == "z") & (R_NOV.pool == "a")]
res = [("claim seed", *stats.ttest_rel(main.late_pct, main.early_pct), (main.late_pct - main.early_pct).mean())]
for k, d in SD_NOV.groupby("seed_offset"):
    res.append((f"seed + {k}", *stats.ttest_rel(d.late_pct, d.early_pct), (d.late_pct - d.early_pct).mean()))
for r in res:
    print(f"{r[0]}: late - early = {r[3]:+.4f} percentage points, t({len(main) - 1}) = {r[1]:.2f}, p = {r[2]:.3g}")
print(f"seeds in {time.time() - t0:.0f} s")
# variant: the 7 sessions with clock-checked tracking (position is not used here)
d = main[main.session != "e16-3m2-211211"]
t, p = stats.ttest_rel(d.late_pct, d.early_pct)
print(f"without e16-3m2-211211: late - early = {(d.late_pct - d.early_pct).mean():+.4f}, t({len(d) - 1}) = {t:.2f}, p = {p:.3g}")

**Result: not reproduced.** The procedure's false positive rate is 0.001 (0.00093 to 0.00107 by session), the
paper's. Under the main reading post ripples reactivate 0.232% of first-minute moments and 0.209% of moments
at 10 to 11 minutes (late - early -0.0236 percentage points, t(7) = -1.40, p = 0.204); under the five seeds the
difference is -0.0148 to -0.0243 (p 0.189 to 0.387). No other reading finds late moments more reactivated at
p < 0.05: raw counts +0.0305 (p = 0.1976), pooled shuffles -0.0108 (p = 0.509) and -0.0167 (p = 0.2911).

## fig8d-familiar-no-difference

> In the familiar environment, early and late moments do not differ in reactivation rate.

Criterion: a claim of absence, judged on the paired t-test over sessions at p < 0.05, on the familiar
figure-eight run with pre and post ripples around it, under E10 (a) with z-scored vectors (pool (a)); the
other readings shown.

In [ ]:
t0 = time.time()
R_FAM = readings("fam", "fig8d-familiar-no-difference")
print(f"familiar readings in {time.time() - t0:.0f} s")
print(R_FAM[R_FAM.set == "post"].round(3).to_string())
LV_FAM = late_vs_early(R_FAM)
print(LV_FAM.round(4).to_string())
SD_FAM = seeds_main("fam", "fig8d-familiar-no-difference")
main = R_FAM[(R_FAM.set == "post") & (R_FAM.pre == "preA") & (R_FAM.vec == "z") & (R_FAM.pool == "a")]
t, p = stats.ttest_rel(main.late_pct, main.early_pct)
print(f"claim seed: late - early = {(main.late_pct - main.early_pct).mean():+.4f}, t({len(main) - 1}) = {t:.2f}, p = {p:.3g}")
for k, d in SD_FAM.groupby("seed_offset"):
    t, p = stats.ttest_rel(d.late_pct, d.early_pct)
    print(f"seed + {k}: late - early = {(d.late_pct - d.early_pct).mean():+.4f}, t({len(d) - 1}) = {t:.2f}, p = {p:.3g}")

**Result: not reproduced.** The judged test finds a difference in the familiar run: early moments are
reactivated more than late ones (0.247% against 0.221%, -0.0255 percentage points, t(7) = -3.72, p = 0.0074),
under all five seeds (p 0.0043 to 0.0074). The other readings find no difference (raw counts p = 0.7383, pooled
shuffles p = 0.1497 and 0.2352). The effect is small, about a tenth of the reactivation rate.

## fig8-no-pre-reactivation

> Track activity patterns show no evidence of being present in ripples recorded before the experience.

Criterion: per session the mean over moments of the percentage of pre ripples that reactivate each moment;
reproduced when its mean over sessions is not above 0.1% by a one-sided one-sample t-test at p < 0.05, in
novel and familiar runs. Main reading: pre (a), z-scored, pool (a), under this claim's own seed; the
other readings come from the runs above.

In [ ]:
t0 = time.time()
rows = []
for k in range(5):
    rng = np.random.default_rng(seed("fig8-no-pre-reactivation") + 1000 * k)
    for env in ("nov", "fam"):
        for sid in TEST:
            for r in run_reactivation(sid, env, rng, "preA", "z", "a", ("pre",)):
                rows.append(dict(r, seed_offset=1000 * k))
PRE = pd.DataFrame(rows)
print(f"pre ripples in {time.time() - t0:.0f} s")
print(PRE[PRE.seed_offset == 0].round(4).to_string())
for (k, env), d in PRE.groupby(["seed_offset", "env"]):
    t, p = stats.ttest_1samp(d.all_pct, 0.1, alternative="greater")
    print(f"seed + {k}, {env}: mean {d.all_pct.mean():.4f}% (sessions {d.all_pct.min():.4f} to {d.all_pct.max():.4f}), one-sided t({len(d) - 1}) = {t:.2f}, p = {p:.3g}")
print("other readings (claim seeds of fig8b and fig8d):")
for env, df in (("nov", R_NOV), ("fam", R_FAM)):
    for key, d in df[df.set == "pre"].groupby(["pre", "vec", "pool"]):
        t, p = stats.ttest_1samp(d.all_pct, 0.1, alternative="greater")
        print(f"  {env} {key}: mean {d.all_pct.mean():.4f}%, t = {t:.2f}, p = {p:.3g}")
    post = df[(df.set == "post") & (df.pre == "preA") & (df.vec == "z") & (df.pool == "a")]
    print(f"  {env} post ripples for comparison: mean {post.all_pct.mean():.4f}%")

**Result: not reproduced.** Pre ripples reactivate on average 0.1334% of moments in the novel runs and 0.1461%
in the familiar runs, above the 0.1% false positive rate (one-sided t(7) = 2.82, p = 0.0129, and 7.90, p =
4.93e-05), under all five seeds (p 0.0111 to 0.0132 and below 0.0001) and under every other reading (p 0.0377 or
less). Post ripples reactivate more (0.2206% and 0.2342%), so the track patterns appear in post ripples more
than in pre ripples, but they are not absent from pre ripples.

## Summary

| Claim | Outcome |
|---|---|
| fig8b-late-moments-reactivated | not reproduced |
| fig8d-familiar-no-difference | not reproduced |
| fig8-no-pre-reactivation | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

Differences from the paper's method: the ripples are those deposited in the files, not detected again from the
LFP (only in the raw files); the moments are those of minute 1 and minutes 10 to 11, not the whole run; the
readings the paper leaves open (pre window, z-scored or raw track vectors, shuffle pool) are run one at a time
around a main reading, not fully crossed. The rates found are low (0.1 to 0.3% of ripples per moment), close to
the false positive rate, so small differences decide these tests. Dataset findings are in `settling.ipynb`.

In [ ]:
print(f"notebook runtime {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")